<a href="https://colab.research.google.com/github/jyizheng/my-study/blob/main/colab/shortestTravelDistance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
class Solution:
    def shortestDistance(self, grid: List[List[int]]) -> int:
        M = len(grid)
        N = len(grid[0])

        points = []
        #barriers = []
        for i in range(M):
            for j in range(N):
                if grid[i][j] == 1:
                    points.append((i,j))
                #elif grid[i][j] == 2:
                #    barriers.append((i,j))


        def bfs(x, y, dist):
            visit = [[False]*N for _ in range(M)]
            visit[x][y] = True
            queue = [(x,y,0)]
            delta = [(1, 0), (-1, 0), (0, 1), (0, -1)]
            dist[x][y] = 0

            while queue:
                x0, y0, d = queue.pop(0)
                for dx, dy in delta:
                    x1, y1 = x0+dx, y0+dy
                    if 0<=x1<M and 0<=y1<N and not visit[x1][y1] and grid[x1][y1] == 0:
                        queue.append((x1,y1, d+1))
                        visit[x1][y1] = True
                        dist[x1][y1] = d+1

        distances = []
        for x,y in points:
            distance = [[-1]*N for _ in range(M)]
            bfs(x, y, distance)
            #print(x, y, distance)
            distances.append(distance)

        #print(distances)

        res = 1e10;
        for i in range(M):
            for j in range(N):
                if grid[i][j] == 0:
                    dist_list = [dist[i][j] for dist in distances]
                    if not -1 in dist_list:
                        #print(dist_list)
                        res = min(res, sum(dist_list))

        if res == 1e10:
            return -1
        else:
            return res





NameError: name 'List' is not defined

有几个明显的优化点和潜在 bug，逐条梳理如下：

---

## 🔴 主要问题

1. **BFS 队列效率低**

   ```python
   queue.pop(0)
   ```

   * 这是 O(n) 操作，会导致性能恶化。
   * 改成 `collections.deque` + `popleft()`，保证 O(1)。

---

2. **BFS 的条件过于严格**

   ```python
   if 0<=x1<M and 0<=y1<N and not visit[x1][y1] and grid[x1][y1] == 0:
   ```

   * 这样写，BFS 只能扩展到空地 (`0`)，**不能穿过建筑 (1)** 或障碍 (2)。
   * 但实际 BFS 是从建筑出发，需要走过空地，**可以在 grid==0 的时候更新，但 grid==1 或 grid==2 不能走进去**。
   * 你这里还写了 `dist[x][y] = 0`，其实建筑本身不需要算距离，只要空地能被所有建筑访问即可。

---

3. **distances 存储开销过大**

   ```python
   distances = []
   for x,y in points:
       distance = [[-1]*N for _ in range(M)]
       bfs(x, y, distance)
       distances.append(distance)
   ```

   * 你为每个建筑存一个全局的 `M*N` 矩阵，假设 M=N=50，有几十个建筑时就很大。
   * 更好的做法：维护一个全局的 `dist_sum` 和 `reach_count`，在每次 BFS 里直接累计，而不是存所有矩阵。

---

4. **最终求解效率低**

   ```python
   dist_list = [dist[i][j] for dist in distances]
   if not -1 in dist_list:
       res = min(res, sum(dist_list))
   ```

   * 这里每个点都要遍历所有建筑的 distance，复杂度 `O(M*N*B)`。
   * 如果在 BFS 阶段就维护好累积距离和可达次数，最后只要 O(M\*N) 遍历一次 grid 就行。

---

## 🔍 改进点总结

1. **`deque` 替代 `list.pop(0)`** → 避免 O(n²)。
2. **不用存每个建筑的完整 distance 矩阵** → 节省空间。
3. **在 BFS 阶段就维护 `dist_sum` 和 `reach_count`** → 避免重复遍历。
4. **最终只遍历一遍 grid** → 提升效率。

复杂度：

* 每个建筑一次 BFS = `O(M*N)`
* 总体 `O(B*M*N)`，这是题目本身的复杂度下限。
* 空间 `O(M*N)`。

---


In [ ]:
from typing import List
from collections import deque

class Solution:
    def shortestDistance(self, grid: List[List[int]]) -> int:
        M, N = len(grid), len(grid[0])
        buildings = [(i,j) for i in range(M) for j in range(N) if grid[i][j] == 1]

        # 记录每个空地的总距离 & 被多少建筑访问过
        dist_sum = [[0]*N for _ in range(M)]
        reach_count = [[0]*N for _ in range(M)]

        # BFS from each building
        def bfs(x, y):
            visited = [[False]*N for _ in range(M)]
            q = deque([(x, y, 0)])
            visited[x][y] = True

            while q:
                i, j, d = q.popleft()
                for dx, dy in [(1,0),(-1,0),(0,1),(0,-1)]:
                    ni, nj = i+dx, j+dy
                    if 0 <= ni < M and 0 <= nj < N and not visited[ni][nj]:
                        if grid[ni][nj] == 0:
                            visited[ni][nj] = True
                            dist_sum[ni][nj] += d+1
                            reach_count[ni][nj] += 1
                            q.append((ni, nj, d+1))
                        elif grid[ni][nj] == 1:
                            visited[ni][nj] = True  # 不继续走，但标记避免重复

        # 从每个建筑 BFS
        for x,y in buildings:
            bfs(x,y)

        # 找最小距离
        res = float("inf")
        for i in range(M):
            for j in range(N):
                if grid[i][j] == 0 and reach_count[i][j] == len(buildings):
                    res = min(res, dist_sum[i][j])

        return res if res != float("inf") else -1
